# sigreg-video-lejepa — Kaggle Linear Probe Notebook

Runs the two-stage linear probe evaluation on UCF101:
1. Feature extraction: run frozen encoder over all videos, save `.pt` tensors to `/kaggle/working/`
2. Probe training: train a single `nn.Linear` on those features, report top-1 / top-5

**Prereqs:**
- UCF101 Kaggle dataset added to this notebook (Settings → Data → Add dataset).
  Expected at `/kaggle/input/ucf101/UCF-101/` (101 class dirs) and
  `/kaggle/input/ucf101/ucfTrainTestlist/` (split files).
- A pretrained checkpoint uploaded as a Kaggle dataset (or use Phase 3 random-init for pipeline testing).

**Workflow:** push code to GitHub → pull here → run cells → save outputs to `/kaggle/working/`.

## 1. Clone repo and install deps

In [ ]:
import os

REPO_DIR = '/kaggle/working/sigreg-video-lejepa'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/ankitpani8/sigreg-video-lejepa.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}
!git log --oneline -5

In [ ]:
!pip install -q uv
!uv pip install --system -e '.[dev]'

## 2. Environment check

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

import os
UCF101_DATA   = '/kaggle/input/ucf101/UCF-101'
UCF101_SPLITS = '/kaggle/input/ucf101/ucfTrainTestlist'
print(f'UCF101 data exists:   {os.path.exists(UCF101_DATA)}')
print(f'UCF101 splits exist:  {os.path.exists(UCF101_SPLITS)}')

## 3. Configure paths

Set `CHECKPOINT_PATH` to a real checkpoint file to evaluate a pretrained encoder.
Leave as `None` to use a randomly-initialized encoder (pipeline smoke test only).

In [ ]:
# Path to a VideoJEPAModule Lightning checkpoint, or None for random-init (Phase 3 test)
CHECKPOINT_PATH = None  # e.g. '/kaggle/input/my-checkpoint/epoch=99.ckpt'

FEATURES_DIR = '/kaggle/working/features'
os.makedirs(FEATURES_DIR, exist_ok=True)
print(f'Features will be saved to: {FEATURES_DIR}')

## 4. Extract features

Runs the encoder over all UCF101 train and test videos (once per checkpoint).
Feature files are written to `FEATURES_DIR`; subsequent cells re-use them.
On a T4 with a T=16 / 64×64 encoder this takes ~30 min for the full dataset.

In [ ]:
ckpt_override = f'checkpoint_path={CHECKPOINT_PATH}' if CHECKPOINT_PATH else 'checkpoint_path=null'

!python scripts/extract_features.py \
    +experiment=ucf101_linprobe \
    {ckpt_override} \
    evaluation.features_dir={FEATURES_DIR} \
    data.dataset.data_root={UCF101_DATA} \
    data.dataset.split_root={UCF101_SPLITS} \
    data.dataset.local_cache=null \
    data.num_workers=2

## 5. Train linear probe

In [ ]:
!python scripts/linear_probe.py \
    +experiment=ucf101_linprobe \
    evaluation.features_dir={FEATURES_DIR} \
    trainer.max_epochs=20 \
    trainer.accelerator=auto

## 6. Display results

In [ ]:
import torch, os
feat_path = os.path.join(FEATURES_DIR, 'test_features.pt')
if os.path.exists(feat_path):
    feats = torch.load(feat_path, weights_only=True)
    labels = torch.load(os.path.join(FEATURES_DIR, 'test_labels.pt'), weights_only=True)
    print(f'Test features shape: {feats.shape}')
    print(f'Test labels shape:   {labels.shape}')
    print(f'Num classes in test: {labels.unique().numel()}')
else:
    print('Feature files not found — run extraction first.')

## 7. Upload to HuggingFace Hub (optional)

Persists features + probe checkpoint to HF Hub so results survive Kaggle session expiry.

In [ ]:
# TODO: set HF_TOKEN before running this cell
# import os
# os.environ['HF_TOKEN'] = 'hf_...'   # paste your token here or set in Kaggle Secrets

# from huggingface_hub import HfApi
# api = HfApi()
# repo_id = 'ankitpani/sigreg-video-lejepa-ucf101-features'
# api.create_repo(repo_id, repo_type='dataset', exist_ok=True)
# for fname in ['train_features.pt', 'train_labels.pt', 'test_features.pt', 'test_labels.pt']:
#     api.upload_file(
#         path_or_fileobj=os.path.join(FEATURES_DIR, fname),
#         path_in_repo=fname,
#         repo_id=repo_id,
#         repo_type='dataset',
#     )
# print(f'Uploaded to https://huggingface.co/datasets/{repo_id}')